- Ative a máquina virtual com:

`source nomeambiente/bin/activate`


- Instalando pacotes e bibliotecas necessárias

In [1]:
%pip install langchain
%pip install langchain-community
%pip install -U langchain-core
%pip install -qU "langchain[openai]"
%pip install langchain-openai


from langchain_community.document_loaders import CSVLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv
import csv
import pandas as pd
import getpass
import os

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Carrega o csv
caminho_arquivo = "./dados_relevantes.csv"

documentos_resultados = []
coluna_id = 'researcher_id'

# Incluí mais colunas que vão ser divididas em mais de uma chunk
colunas_com_textos_longos = ['abstract', 'articles', 'description_project', 'project_name']

CHUNK_SIZE = 700
CHUNK_OVERLAP = 100

# Personalizei a ordem dos separadores como achei melhor
text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", "; ", ". ", ", "],
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    add_start_index=True
)

with open(caminho_arquivo, mode='r', encoding='utf-8') as arquivo_csv:
    # dictReader vai ler cada linha como um dicionário (coluna -> valor)
    leitor_csv = csv.DictReader(arquivo_csv)
    
    # itera sobre cada linha do csv
    for i, linha in enumerate(leitor_csv):
        id_pesquisador = linha[coluna_id]
        
        for nome_coluna, valor_coluna in linha.items():
            
            # Eliminei os chunks das colunas sem registro
            if valor_coluna == 'Sem registro':
                continue
            
            # --- Lógica para colunas de texto longo ---
            elif nome_coluna in colunas_com_textos_longos:
                
                chunks_como_documentos = text_splitter.create_documents([valor_coluna])
              
                for j, doc_chunk in enumerate(chunks_como_documentos):
                    # Analisar se fica melhor contextualizando com nome da coluna mesmo <----
                    doc_chunk.page_content = f"Do documento com ID '{id_pesquisador}', um trecho da coluna '{nome_coluna}' é: {doc_chunk.page_content}"
                    
                    doc_chunk.metadata.update({
                        'source': caminho_arquivo,
                        'researcher_id': id_pesquisador,
                        'row': i,
                        'column': nome_coluna,
                        'chunk_index': j 
                    })
                    documentos_resultados.append(doc_chunk)

            # --- Lógica para colunas curtas ---
            else:
                # para não repetir linha
                if nome_coluna != coluna_id:
                    conteudo = f"Do documento com ID '{id_pesquisador}', a informação de '{nome_coluna}' é: '{valor_coluna}'"
                    metadata = {
                        'source': caminho_arquivo,
                        'researcher_id': id_pesquisador,
                        'row': i,
                        'column': nome_coluna
                    }
                    doc = Document(page_content=conteudo, metadata=metadata)
                    documentos_resultados.append(doc)

print(f"\nProcessamento finalizado. Total de {len(documentos_resultados)} documentos (chunks) gerados.")
print("-" * 50)

'''# --- Visualização de Resultados ---''
print("Visualizando alguns dos chunks gerados para ver a diferença:\n")

for doc in documentos_resultados[:80]:
    e_chunk_de_texto_longo = 'chunk_index' in doc.metadata
    
    if e_chunk_de_texto_longo:
        print("\x1b[36m--- Chunk de Texto Longo --- \x1b[0m")
    else:
        print("\x1b[32m--- Chunk de Coluna Curta --- \x1b[0m")

    print(f"Conteúdo: {doc.page_content}")
    print(f"(Tamanho do conteúdo: {len(doc.page_content)} caracteres)")
    print(f"Metadados: {doc.metadata}\n") '''




Processamento finalizado. Total de 8840 documentos (chunks) gerados.
--------------------------------------------------


'# --- Visualização de Resultados ---\'\'\nprint("Visualizando alguns dos chunks gerados para ver a diferença:\n")\n\nfor doc in documentos_resultados[:80]:\n    e_chunk_de_texto_longo = \'chunk_index\' in doc.metadata\n\n    if e_chunk_de_texto_longo:\n        print("\x1b--- Chunk de Texto Longo --- \x1b")\n    else:\n        print("\x1b--- Chunk de Coluna Curta --- \x1b")\n\n    print(f"Conteúdo: {doc.page_content}")\n    print(f"(Tamanho do conteúdo: {len(doc.page_content)} caracteres)")\n    print(f"Metadados: {doc.metadata}\n") '

In [ ]:
print("\nIniciando a geração de embeddings com a OpenAI...")

# Carrega as variáveis do arquivo .env para o ambiente
load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("A variável de ambiente OPENAI_API_KEY não foi encontrada. Crie um arquivo .env ou configure a variável de ambiente.")

try:
    embeddings_model = OpenAIEmbeddings()

    # Extrai o conteúdo e os IDs de cada documento para processamento em lote (mais eficiente)
    conteudos_dos_chunks = [doc.page_content for doc in documentos_resultados]
    ids_dos_pesquisadores = [doc.metadata['researcher_id'] for doc in documentos_resultados]

    # Gera os embeddings para todos os chunks de uma vez
    vetores_embeddings = embeddings_model.embed_documents(conteudos_dos_chunks)

    print(f"✔️ {len(vetores_embeddings)} embeddings gerados com sucesso.")

    df = pd.DataFrame({
        'id_pesquisador': ids_dos_pesquisadores,
        'embeddings': vetores_embeddings
    })

    print("\n--- DataFrame Criado ---")
    print(f"\nDimensões do DataFrame: {df.shape}")

    if not df.empty:
        # Cria uma cópia da primeira linha para não alterar o DataFrame original
        df_amostra = df.head(1).copy()
    # Ver como melhorar pra visualizar e consertar relação id com embeddings no dataframe
        print(df_amostra)

except Exception as e:
    print("\n\033[91m❌ Ocorreu um erro ao gerar os embeddings ou criar o DataFrame.\033[0m")
    print("Verifique se a sua chave da API da OpenAI está configurada corretamente no seu arquivo .env.")
    print(f"Erro: {e}")


Iniciando a geração de embeddings com a OpenAI...
✔️ 8840 embeddings gerados com sucesso.

--- DataFrame Criado ---

Dimensões do DataFrame: (8840, 2)
                         id_pesquisador  \
0  00d79509-2e3a-4ef9-95f6-f9d5c79c34cf   

                                          embeddings  
0  [-0.02257746458053589, 0.02746168151497841, -0...  
